In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 5000

df = pd.DataFrame({
    "loan_id": range(1, n+1),
    "partner": np.random.choice(["A", "B", "C", "D"], n),
    "loan_amount": np.random.randint(20000, 500000, n),
    "tenure_months": np.random.choice([12, 24, 36, 48, 60], n),
    "credit_score": np.random.randint(550, 850, n),
    "status": np.random.choice(["active", "closed", "defaulted"], n, p = [0.5, 0.4, 0.1])
})

df.to_csv("loans.csv", index = False)
print("Saved", len(df), "rows")

Saved 5000 rows


In [2]:
import duckdb

con = duckdb.connect()

con.execute("CREATE TABLE loans AS SELECT * FROM read_csv_auto('loans.csv')")

result = con.execute("""
    SELECT partner, COUNT(*) AS n, AVG(loan_amount) AS avg_amt
    FROM loans
    GROUP BY partner
""").df()

print(result)

  partner     n        avg_amt
0       A  1295  265855.700386
1       B  1240  265310.995161
2       C  1209  266977.693962
3       D  1256  261013.792994


In [3]:
import getpass
from google import genai
from google.genai import types

client = genai.Client(api_key=getpass.getpass("Gemini API key: "))

In [4]:
schema = """
Table: loans
Columns:
- loan_id (int)
- partner (text: A, B, C, D)
- loan_amount (int)
- tenure_months (int: 12, 24, 36, 48, 60)
- credit_score (int)
- status (text: active, closed, defaulted)
"""

def question_to_sql(question):
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=f"You write DuckDB SQL. Use only this schema:\n{schema}\nReturn only the SQL query, with no explanation and no markdown."
        )
    )
    return response.text.strip()

In [5]:
sql = question_to_sql("What is the average loan amount of each partner?")
print(sql)

SELECT partner, AVG(loan_amount) FROM loans GROUP BY partner;


In [6]:
con.execute(sql).df()

,partner,avg(loan_amount)
0,A,265855.700386
1,B,265310.995161
2,C,266977.693962
3,D,261013.792994


In [7]:
con.execute("""
    SELECT partner,
    AVG(CASE WHEN status = 'defaulted' THEN 1.0 ELSE 0 END) AS default_rate
    FROM loans
    GROUP BY partner
    ORDER BY default_rate DESC
""").df()

,partner,default_rate
0,D,0.117834
1,C,0.098428
2,A,0.096525
3,B,0.091129


In [8]:
def is_safe(sql):
    s = sql.strip().rstrip(";").lower()
    banned = ["insert", "update", "delete", "drop", "alter", "create", "attach", "copy", "pragma"]
    return s.startswith("select") and ";" not in s and not any(w in s for w in banned)

def clean_sql(sql):
    return sql.replace("```sql", "").replace("```", "").strip()

def ask(question):
    sql = clean_sql(question_to_sql(question))
    if not is_safe(sql):
        return sql, None, "Blocked: only read-only SELECT queries are allowed"

    try:
        df = con.execute(sql).df()
    except Exception as e:
        return sql, None, f"The generated SQL failed: {e}"

    summary = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = f"Question: {question}\nSQL result:\n{df.head(20).to_string()}\nAnswer in 2-3 plain-English sentences."
    ).text

    return sql, df, summary

In [9]:
sql, df, summary = ask("Average loan amount by tenure for active loans")
print(sql)
display(df)
print(summary)

SELECT tenure_months, AVG(loan_amount) FROM loans WHERE status = 'active' GROUP BY tenure_months;


,tenure_months,avg(loan_amount)
0,12,266366.390963
1,24,256755.500000
2,36,261705.804082
3,48,255488.691837
4,60,261639.869888


The average amount for active loans remains relatively consistent across different terms, ranging between approximately $255,000 and $266,000. Loans with a 12-month tenure carry the highest average balance at $266,366, while those with a 48-month tenure have the lowest average at $255,489.
